# 4주차 보충 학습 · 이진 분류, 다중 분류, 회귀의 출력층·Loss·Metrics

같은 신경망이라도 **무엇을 예측하는지**에 따라 마지막 출력층, 활성화 함수, loss, metrics가 달라집니다. 이 노트는 설정을 외우는 데서 끝내지 않고 각 항목이 서로 맞물리는 이유를 코드로 확인합니다.

학습 목표

- `loss`와 `metrics`의 역할을 구분한다.
- 이진 분류·다중 분류·회귀의 출력 형태에 맞는 설정을 선택한다.
- 회귀에서 활성화 함수가 없다는 말과 loss가 없다는 말을 구분한다.

## 1. Loss와 Metrics는 역할이 다릅니다

| 구분 | 역할 | 가중치 학습에 직접 사용되는가? |
|---|---|---|
| `loss` | 예측과 정답의 차이를 하나의 값으로 계산 | **예**. 모델은 이 값을 줄이도록 학습 |
| `metrics` | 사람이 모델 성능을 이해하기 쉽게 표시 | **아니요**. 기본적으로 학습 경과를 관찰하는 용도 |

지도학습에서는 학습 기준인 loss가 필요합니다. `metrics`는 선택 사항이므로 생략할 수 있으며, 생략하면 학습 화면에 loss만 표시됩니다.

In [ ]:
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

## 2. 이진 분류

이진 분류는 두 범주 중 하나를 예측합니다. 출력 뉴런 하나와 `sigmoid`를 사용하면 모델은 **1번 클래스일 확률 하나**를 0~1 사이로 출력합니다. 정답도 0 또는 1이므로 베르누이 확률의 오차를 계산하는 `binary_crossentropy`와 맞습니다.

`accuracy`는 보통 확률이 0.5 이상인지로 클래스를 정한 뒤 맞은 비율을 계산합니다. 클래스 불균형이 크다면 precision, recall, AUC도 함께 확인해야 합니다.

In [ ]:
binary_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),  # P(y=1) 하나
])
binary_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

binary_probability = binary_model(np.zeros((2, 4)), training=False)
print("출력 shape:", binary_probability.shape)  # (샘플 수, 1)
print("예측 클래스:", (binary_probability.numpy() >= 0.5).astype(int).ravel())

## 3. 다중 분류

다중 분류는 여러 클래스 중 **하나**를 예측합니다. 클래스 수만큼 출력 뉴런을 만들고 `softmax`로 각 클래스의 확률을 구하면 모든 확률의 합은 1이 됩니다. Cross-entropy는 정답 클래스에 높은 확률을 주었는지를 평가합니다.

정답 표현에 따라 loss 이름이 달라집니다.

| 정답 형태 | 예시 | Loss |
|---|---|---|
| 원-핫 벡터 | `[0, 1, 0]` | `categorical_crossentropy` |
| 정수 클래스 번호 | `1` | `sparse_categorical_crossentropy` |

정수 라벨을 sparse categorical cross-entropy에 전달하면 숫자의 크기나 순서를 비교하지 않고 **클래스 번호**로 처리합니다. 여러 클래스를 동시에 선택하는 다중 레이블 분류는 별개의 문제이며, 보통 클래스별 sigmoid와 binary cross-entropy를 사용합니다.

In [ ]:
multiclass_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),  # 클래스별 확률 3개
])
multiclass_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  # 정답이 0, 1, 2인 경우
    metrics=["accuracy"],
)

class_probabilities = multiclass_model(np.zeros((2, 4)), training=False)
print("출력 shape:", class_probabilities.shape)  # (샘플 수, 3)
print("확률 합:", np.sum(class_probabilities.numpy(), axis=1))
print("예측 클래스:", np.argmax(class_probabilities.numpy(), axis=1))

## 4. 회귀

회귀는 집값, 온도처럼 연속된 수치를 예측합니다. 마지막 층의 활성화 함수를 생략하면 Keras의 기본값인 **선형(linear) 활성화**가 적용되어 출력 범위를 0~1로 제한하지 않습니다. 출력층이 없어지는 것은 아닙니다.

- `mse`: 오차를 제곱하므로 큰 오차에 더 큰 벌점을 줍니다. 미분하기 쉬워 회귀 loss로 자주 사용합니다.
- `mae`: 평균적으로 실제값과 얼마나 차이 나는지 원래 단위로 해석하기 쉽습니다. 회귀 metric으로 자주 사용합니다.

MSE만 가능한 것은 아닙니다. 이상치에 덜 민감하게 학습하려면 MAE나 Huber loss를 선택할 수도 있습니다.

In [ ]:
regression_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1),  # activation=None은 linear와 같음
])
regression_model.compile(
    optimizer="adam",
    loss="mse",       # 학습에서 실제로 최소화
    metrics=["mae"],  # 사람이 평균 오차 크기를 확인
)

continuous_value = regression_model(np.zeros((2, 4)), training=False)
print("출력 shape:", continuous_value.shape)  # (샘플 수, 1)
print("연속값 예측:", continuous_value.numpy().ravel())

### 회귀에서 Metrics를 생략할 수 있습니다

아래 모델에도 loss는 있지만 metrics는 없습니다. 이 경우에도 학습은 정상적으로 진행되고 loss만 기록됩니다. 같은 방식으로 분류 모델에서도 metrics를 생략할 수 있습니다.

In [ ]:
regression_model_without_metric = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(1),
])
regression_model_without_metric.compile(
    optimizer="adam",
    loss="mse",
)
print("표시할 metrics:", regression_model_without_metric.metrics_names)

## 5. 한눈에 선택하기

| 문제 유형 | 정답 예시 | 마지막 층 | 대표 Loss | 대표 Metrics | 예측 처리 |
|---|---|---|---|---|---|
| 이진 분류 | `0` 또는 `1` | `Dense(1, sigmoid)` | `binary_crossentropy` | accuracy, precision, recall, AUC | 확률에 임계값 적용 |
| 다중 분류(원-핫) | `[0, 1, 0]` | `Dense(C, softmax)` | `categorical_crossentropy` | accuracy | `argmax` |
| 다중 분류(정수) | `1` | `Dense(C, softmax)` | `sparse_categorical_crossentropy` | accuracy | `argmax` |
| 회귀 | `23.7` | `Dense(1)` | MSE, MAE, Huber | MAE, RMSE | 출력값 그대로 사용 |

선택 순서

1. 정답이 범주인지 연속값인지 확인합니다.
2. 범주라면 두 개인지, 여러 개 중 하나인지, 여러 개를 동시에 고르는지 확인합니다.
3. 원하는 예측값의 형태에 맞춰 출력층과 활성화 함수를 정합니다.
4. 출력값과 정답 형식을 비교할 수 있는 loss를 고릅니다.
5. 사람이 해석할 성능 지표를 metrics에 추가합니다.

## 6. 5주차 Wine 실습과 연결

Wine은 세 품종 중 하나를 고르는 다중 분류입니다. 5주차 실습에서 정답을 원-핫 벡터로 변환했으므로 다음 조합을 사용합니다.

```python
Dense(3, activation="softmax")
loss="categorical_crossentropy"
metrics=["accuracy"]
```

정답을 0, 1, 2의 정수로 유지한다면 출력층은 그대로 두고 loss만 `sparse_categorical_crossentropy`로 바꾸면 됩니다. 테스트 loss를 `model.evaluate`로 계산할 때도 학습에 사용한 loss와 정답 형식이 맞아야 합니다.

## 7. 최종 확인

- loss는 모델이 줄이는 값이고 metrics는 사람이 관찰하는 값입니다.
- 회귀의 `Dense(1)`에도 선형 활성화가 적용되며, 학습을 위한 loss가 필요합니다.
- MSE와 MAE는 역할에 따라 loss와 metric 어느 쪽에도 사용할 수 있습니다.
- 출력층, 정답 형식, loss는 서로 맞는 조합이어야 합니다.